# 1. Configure Event Hubs Connection

Retrieve the Event Hubs connection string from the Databricks secret scope and configure Spark's Kafka source.

- Secret scope: `banking-fraud-detect`
- Secret key: `event-hubs-connection-string`
- Event Hub: `banking-transactions`
- Namespace: `ehns-realtime-banking-fraud-detection`

The connection string is retrieved from Databricks Secrets and must never be printed or committed to source control.

In [0]:
connection_string = dbutils.secrets.get(
    scope="banking-fraud-detect",
    key="event-hubs-connection-string"
)

print("Event Hubs secret retrieved successfully.")

Event Hubs secret retrieved successfully.


# 2. Test Event Hubs Connectivity

This diagnostic reads events from the Event Hubs Kafka endpoint and writes them to the notebook console.

Use it when troubleshooting connectivity. It is not the production ingestion query. Because the source uses `startingOffsets = latest`, it reads newly arriving events rather than replaying all earlier events

In [0]:
connection_string = dbutils.secrets.get(
    scope="banking-fraud-detect",
    key="event-hubs-connection-string"
).strip()

namespace = "ehns-realtime-banking-fraud-detection.servicebus.windows.net"

kafka_options = {
    "kafka.bootstrap.servers": f"{namespace}:9093",
    "subscribe": "banking-transactions",
    "startingOffsets": "latest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        'username="$ConnectionString" '
        f'password="{connection_string}";'
    )
}

print("Kafka configuration refreshed.")

Kafka configuration refreshed.


### verify the Event Hubs connection before starting the stream.
### Test Kafka connectivity

In [0]:
connection_string = dbutils.secrets.get(
    scope="banking-fraud-detect",
    key="event-hubs-connection-string"
).strip()

print("Starts with Endpoint=sb://:", connection_string.startswith("Endpoint=sb://"))
print("Contains SharedAccessKeyName:", "SharedAccessKeyName=" in connection_string)
print("Contains SharedAccessKey:", "SharedAccessKey=" in connection_string)
print("Contains EntityPath:", "EntityPath=" in connection_string)

Starts with Endpoint=sb://: True
Contains SharedAccessKeyName: True
Contains SharedAccessKey: True
Contains EntityPath: False


In [0]:
connection_string = dbutils.secrets.get(
    scope="banking-fraud-detect",
    key="event-hubs-connection-string"
).strip()

print("Endpoint:", connection_string.startswith("Endpoint=sb://"))
print("Key name:", "SharedAccessKeyName=" in connection_string)
print("Key value present:", "SharedAccessKey=" in connection_string)

Endpoint: True
Key name: True
Key value present: True


# 3. Inspect Incoming Events

Display the event payload as a string along with its Kafka timestamp.

Use this cell to confirm that transactions are arriving and to inspect the raw JSON. Stop the preview query when you no longer need it, and avoid running unnecessary streaming queries alongside the Bronze ingestion query.

In [0]:
test_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
)

query = (
    test_stream.writeStream
    .format("console")
    .option(
        "checkpointLocation",
        "abfss://bronze@stbankingfrauddetection.dfs.core.windows.net/checkpoints/eventhub_connectivity_test_v3/"
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
from pyspark.sql.functions import col

inspect_stream = (
    test_stream
    .select(
        col("value").cast("string").alias("transaction_json"),
        col("timestamp").alias("event_timestamp")
    )
)

display(
    inspect_stream,
    checkpointLocation="abfss://bronze@stbankingfrauddetection.dfs.core.windows.net/checkpoints/eventhub_inspect_v3/"
)

transaction_json,event_timestamp
"{""transaction_id"": ""566502e0-bfeb-4bdf-9e3a-88193edf0151"", ""account_id"": ""ACC1019"", ""amount"": 3761.43, ""merchant"": ""Electronics Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:04.111867+00:00"", ""device_id"": ""DEV120"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:04.167Z
"{""transaction_id"": ""a9ee3f06-26e3-4f38-bf3e-bcaba3996a71"", ""account_id"": ""ACC1006"", ""amount"": 9802.52, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:08.270633+00:00"", ""device_id"": ""DEV107"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:08.325Z
"{""transaction_id"": ""a7022094-3465-490c-91ed-d6d7439cb11a"", ""account_id"": ""ACC1020"", ""amount"": 4551.76, ""merchant"": ""Electronics Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:06.191129+00:00"", ""device_id"": ""DEV112"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:06.241Z
"{""transaction_id"": ""cd7fc0ef-c94e-4dc9-8db8-3a63a0527932"", ""account_id"": ""ACC1008"", ""amount"": 2553.87, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:10.349549+00:00"", ""device_id"": ""DEV115"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:10.398Z
"{""transaction_id"": ""255d881a-db76-42d5-8de4-8c9bdeef7c5a"", ""account_id"": ""ACC1017"", ""amount"": 9036.47, ""merchant"": ""Fuel Station"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:12.429926+00:00"", ""device_id"": ""DEV107"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:12.481Z
"{""transaction_id"": ""a99698b8-c1ba-47c2-a306-468862d187f2"", ""account_id"": ""ACC1011"", ""amount"": 9876.08, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:14.508350+00:00"", ""device_id"": ""DEV100"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:14.555Z
"{""transaction_id"": ""24520a5b-76c8-49f5-9f6c-0c17e1cae3ee"", ""account_id"": ""ACC1016"", ""amount"": 5563.01, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:16.586532+00:00"", ""device_id"": ""DEV111"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:16.639Z
"{""transaction_id"": ""907f6684-e8aa-4810-81b7-f47cba625e61"", ""account_id"": ""ACC1013"", ""amount"": 5009.0, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:18.666276+00:00"", ""device_id"": ""DEV107"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:18.712Z
"{""transaction_id"": ""b5f0294f-610b-4422-9346-d99f760247a2"", ""account_id"": ""ACC1017"", ""amount"": 2712.92, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:20.743226+00:00"", ""device_id"": ""DEV106"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:20.798Z
"{""transaction_id"": ""a19406b4-a021-41f9-90c3-cf01f7d2eb8a"", ""account_id"": ""ACC1018"", ""amount"": 9814.11, ""merchant"": ""Restaurant"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:16:22.820363+00:00"", ""device_id"": ""DEV118"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",2026-10-20T14:16:22.869Z


# 4. Bronze Layer — Raw Event Ingestion

Read transaction events from Azure Event Hubs and append them to a Delta dataset in the ADLS Gen2 `bronze` container.

The Bronze dataset preserves:
- Raw transaction JSON
- Kafka topic, partition, and offset
- Kafka event timestamp
- Databricks ingestion timestamp

A dedicated checkpoint path stores streaming progress so the query can recover after a restart. Keep the stream running while the Python transaction producer is sending events.

In [0]:
from pyspark.sql.functions import col, current_timestamp

bronze_path = (
    "abfss://bronze@stbankingfrauddetection.dfs.core.windows.net/"
    "banking_transactions/"
)

checkpoint_path = (
    "abfss://bronze@stbankingfrauddetection.dfs.core.windows.net/"
    "checkpoints/bronze_ingestion_v1/"
)

bronze_df = test_stream.select(
    col("value").cast("string").alias("transaction_json"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp").alias("kafka_timestamp"),
    current_timestamp().alias("ingestion_timestamp")
)

bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("path", bronze_path)
    .option("checkpointLocation", checkpoint_path)
    .start()
)

print("Bronze ingestion stream started.")

Bronze ingestion stream started.


# 5. Verify Bronze Delta Data

Read the Bronze Delta dataset from ADLS Gen2 and display a small sample to confirm that events have been persisted.

Run this after the Bronze ingestion query has written data. This is a batch read for verification; it does not start another streaming query.

In [0]:
display(
    spark.read.format("delta")
    .load("abfss://bronze@stbankingfrauddetection.dfs.core.windows.net/banking_transactions/")
    .limit(10)
)

transaction_json,topic,partition,offset,kafka_timestamp,ingestion_timestamp
"{""transaction_id"": ""97d687cb-0b98-4656-b69a-1c71c3871c03"", ""account_id"": ""ACC1007"", ""amount"": 3408.78, ""merchant"": ""Electronics Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:58:36.829111+00:00"", ""device_id"": ""DEV101"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",banking-transactions,0,759,2026-10-19T07:58:36.896Z,2026-10-19T07:58:38.185Z
"{""transaction_id"": ""d06156b9-ff65-4dd0-af45-eef6f1942ffd"", ""account_id"": ""ACC1002"", ""amount"": 9335.99, ""merchant"": ""Electronics Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:49:42.531713+00:00"", ""device_id"": ""DEV104"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",banking-transactions,0,639,2026-10-19T07:49:42.855Z,2026-10-19T07:49:44.88Z
"{""transaction_id"": ""4d72e3d5-6035-404f-a36c-41115ba008b9"", ""account_id"": ""ACC1002"", ""amount"": 3673.48, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:12:54.249799+00:00"", ""device_id"": ""DEV106"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",banking-transactions,0,432,2026-10-19T07:12:54.341Z,2026-10-19T07:12:54.543Z
"{""transaction_id"": ""3c97c5a5-0f1e-490e-aefa-6f64a785befa"", ""account_id"": ""ACC1019"", ""amount"": 7567.39, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:47:27.221149+00:00"", ""device_id"": ""DEV115"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",banking-transactions,1,608,2026-10-19T07:47:27.881Z,2026-10-19T07:47:28.501Z
"{""transaction_id"": ""dfdd6342-25db-444f-a8a3-522a84a79c67"", ""account_id"": ""ACC1012"", ""amount"": 5060.0, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:08:58.307081+00:00"", ""device_id"": ""DEV101"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",banking-transactions,1,379,2026-10-19T07:08:58.452Z,2026-10-19T07:08:58.969Z
"{""transaction_id"": ""ab5e30cd-359d-4072-a2ef-484b5e5e2f2f"", ""account_id"": ""ACC1002"", ""amount"": 80336.15, ""merchant"": ""Restaurant"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:57:35.342431+00:00"", ""device_id"": ""DEV113"", ""is_fraud"": true, ""transaction_type"": ""HIGH_VALUE""}",banking-transactions,1,744,2026-10-19T07:57:35.425Z,2026-10-19T07:57:36.789Z
"{""transaction_id"": ""03c5db21-5de4-4a10-a7a1-7d88c123b4d7"", ""account_id"": ""ACC1008"", ""amount"": 8955.69, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:39:30.959625+00:00"", ""device_id"": ""DEV112"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",banking-transactions,1,499,2026-10-19T07:39:31.676Z,2026-10-19T07:39:32.702Z
"{""transaction_id"": ""c78f76b3-4036-4257-a747-c9d0fb92e32c"", ""account_id"": ""ACC1005"", ""amount"": 6678.76, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:43:43.457639+00:00"", ""device_id"": ""DEV120"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",banking-transactions,0,558,2026-10-19T07:43:43.583Z,2026-10-19T07:43:45.275Z
"{""transaction_id"": ""7f754934-5ca4-4b07-a066-03d8a38285b1"", ""account_id"": ""ACC1005"", ""amount"": 4707.65, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:09:27.130938+00:00"", ""device_id"": ""DEV120"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",banking-transactions,0,385,2026-10-19T07:09:27.231Z,2026-10-19T07:09:28.035Z
"{""transaction_id"": ""209bccee-51e1-4781-863d-20d8c053cbe7"", ""account_id"": ""ACC1009"", ""amount"": 1425.95, ""merchant"": ""Fuel Station"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-16T09:10:52.889547+00:00"", ""device_id"": ""DEV120"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"

# 6. Stop Bronze Ingestion (Run Only When Needed)

Run the following control cell only when you intentionally want to stop the active Bronze streaming query.

Stopping this query pauses ingestion. It does not delete the Bronze Delta data or checkpoint files.

In [0]:
bronze_query.stop()

# 7. Inspect Raw JSON Before Silver Transformation

Display a small sample of the raw transaction JSON stored in Bronze. Use the actual field names and data types in this output to define the Silver schema.

This is a one-time inspection step; it does not transform or modify the Bronze data.

In [0]:
display(
    spark.read.format("delta")
    .load("abfss://bronze@stbankingfrauddetection.dfs.core.windows.net/banking_transactions/")
    .select("transaction_json")
    .limit(5)
)

transaction_json
"{""transaction_id"": ""d06156b9-ff65-4dd0-af45-eef6f1942ffd"", ""account_id"": ""ACC1002"", ""amount"": 9335.99, ""merchant"": ""Electronics Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:49:42.531713+00:00"", ""device_id"": ""DEV104"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"
"{""transaction_id"": ""4d72e3d5-6035-404f-a36c-41115ba008b9"", ""account_id"": ""ACC1002"", ""amount"": 3673.48, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:12:54.249799+00:00"", ""device_id"": ""DEV106"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"
"{""transaction_id"": ""dfdd6342-25db-444f-a8a3-522a84a79c67"", ""account_id"": ""ACC1012"", ""amount"": 5060.0, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:08:58.307081+00:00"", ""device_id"": ""DEV101"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"
"{""transaction_id"": ""03c5db21-5de4-4a10-a7a1-7d88c123b4d7"", ""account_id"": ""ACC1008"", ""amount"": 8955.69, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:39:30.959625+00:00"", ""device_id"": ""DEV112"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"
"{""transaction_id"": ""c78f76b3-4036-4257-a747-c9d0fb92e32c"", ""account_id"": ""ACC1005"", ""amount"": 6678.76, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:43:43.457639+00:00"", ""device_id"": ""DEV120"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"
